# 1. set-covering
* 모든 수요지를 커버할 수 있는 후보지의 최적 개수 구하기 = p값 구하기

In [ ]:
pip install pulp

In [ ]:
#set-covering
import pandas as pd
from geopy.distance import geodesic
from itertools import combinations
from pulp import LpProblem, LpVariable, LpBinary, lpSum, LpMinimize, value

In [ ]:
# 엑셀 파일에서 데이터 읽기
demand_df = pd.read_excel(r'수요지_우체통_반경제거.xlsx')  # 수요지 데이터
candidates_df = pd.read_excel(r'후보지.xlsx')  # 후보지 데이터
distance_df = pd.read_excel(r'distance_matrix_TMAP(1-196).xlsx')  # 거리 데이터

In [ ]:
# 수요지와 후보지의 위도 경도 정보 추출
demand_locations = demand_df[['위도', '경도']].values
candidate_locations = candidates_df[['위도', '경도']].values

In [ ]:
# Set-covering 문제 생성
prob = LpProblem("Set Covering Problem", LpMinimize)

# 수요지를 커버하는 후보지 선택 변수
cover_vars = LpVariable.dicts("Cover", range(len(demand_locations)), cat=LpBinary)

# 목적 함수: 후보지 선택 비용 최소화
prob += lpSum(cover_vars)

# 모든 수요지가 커버되어야 함
for i, demand_loc in enumerate(demand_locations):
    covered = []
    for j, candidate_loc in enumerate(candidate_locations):
        if  distance_df.loc[i][j] <= 500:
            covered.append(cover_vars[j])
    prob += lpSum(covered) >= 1

# 최적화 실행
prob.solve()

# 최적의 후보지 개수
optimal_candidate_count = sum([value(cover_vars[j]) for j in range(len(candidate_locations)) if value(cover_vars[j]) > 0.5])
print("Optimal number of candidate locations (Set Covering):", optimal_candidate_count)

# 선택된 후보지 출력
selected_candidates = [j for j in range(len(candidate_locations)) if value(cover_vars[j]) > 0.5]
print("Selected candidate locations (Set Covering):")
for j in selected_candidates:
    print(f"Candidate Location {j+1}: {candidate_locations[j]}")

# 2. set-covering 시각화
* 후보지: 파란색
* 선택된 후보지: 빨간색
* 수요지: 초록색

In [ ]:
# set-covering 시각화
!pip install gmplot

In [ ]:
import gmplot
# 선택된 후보지, 선택되지 않는 후보지, 수요지를 각각 다른 색으로 표시
gmap = gmplot.GoogleMapPlotter(demand_locations[0][0], demand_locations[0][1], 13)

# 선택된 후보지 표시
for j in range(len(candidate_locations)):
    if value(cover_vars[j]) > 0.5:
        gmap.marker(candidate_locations[j][0], candidate_locations[j][1], color='red', title=f"Candidate {j+1} (Selected)")
    else:
        gmap.marker(candidate_locations[j][0], candidate_locations[j][1], color='blue', title=f"Candidate {j+1}")

# 수요지 표시
for i in range(len(demand_locations)):
    gmap.marker(demand_locations[i][0], demand_locations[i][1], color='green', title=f"Demand {i+1}")

# 범례 추가
gmap.marker(0, 0, color='blue', title='Candidate')
gmap.marker(0, 0, color='red', title='Selected Candidate')
gmap.marker(0, 0, color='green', title='Demand')

# HTML 파일로 저장
gmap.draw("map_set-covering4.html")

# 3. 수요가중치 설정
* 각 수요지의 세대수를 수요가중치로 설정

In [ ]:
weights_df = pd.read_excel(r'수요지_우체통_반경제거_세대수(1-148).xlsx')
demand_weights = weights_df['세대 수'].values

# 4. p-median
* 폐의약품 수거함의 최적 위치 선정

In [ ]:
# 후보지와 수요지의 좌표를 튜플 리스트로 변환
candidate_coords = list(zip(candidate_locations[:, 0], candidate_locations[:, 1]))
demand_coords = list(zip(demand_locations[:, 0], demand_locations[:, 1]))

# p-median 문제 정의
p = 6  # 선택할 후보지의 수
prob_pm = LpProblem("P-Median_Problem", LpMinimize)

# 결정 변수 생성
use_location = LpVariable.dicts("Use_Location", range(len(candidate_coords)), cat='Binary')
assign = LpVariable.dicts("Assign", [(i, j) for i in range(len(demand_coords)) for j in range(len(candidate_coords))], cat='Binary')


# 목적 함수: 수요지에서 선택된 후보지까지의 거리 최소화
prob_pm += lpSum(demand_weights[i] * distance_df[j][i] * assign[(i, j)] for i in range(len(demand_coords)) for j in range(len(candidate_coords)))

# 제약 조건: 정확히 p개의 후보지만 사용
prob_pm += lpSum(use_location[j] for j in range(len(candidate_coords))) == p

# 제약 조건: 각 수요지는 하나의 후보지에만 할당
for i in range(len(demand_coords)):
    prob_pm += lpSum(assign[(i, j)] for j in range(len(candidate_coords))) == 1

# 제약 조건: 수요지는 선택된 후보지에만 할당 가능
for i in range(len(demand_coords)):
    for j in range(len(candidate_coords)):
        prob_pm += assign[(i, j)] <= use_location[j]

# 문제를 풀고 결과 출력
prob_pm.solve()

# 선택된 후보지 출력
selected_locations_pm = [j for j in range(len(candidate_coords)) if use_location[j].value() == 1]
print("Selected candidate locations (P-Median):", selected_locations_pm)

# 각 수요지에 할당된 후보지 출력
for i in range(len(demand_coords)):
    for j in selected_locations_pm:
        if assign[(i, j)].value() == 1:
            print(f"Demand point {i+1} is assigned to candidate location {j}.")

# 5. p-median 시각화

In [ ]:
#pip install folium

In [ ]:
import requests
import json
import re
from json import loads
import folium
import pandas as pd
from pyproj import Proj, transform
import folium
from pyproj import Transformer
import numpy as np

In [ ]:
import pandas as pd

# 수요지와 후보지의 위도 경도 정보를 포함한 리스트를 생성
results = []

# 각 수요지에 할당된 후보지 정보를 추가
for i in range(len(demand_coords)):
    for j in selected_locations_pm:
        if assign[(i, j)].value() == 1:
            # 수요지와 후보지의 위도 경도 정보를 가져오기
            demand_lat, demand_lon = demand_coords[i]
            candidate_lat, candidate_lon = candidate_coords[j]
            results.append({
                '수요지 번호': i + 1,
                '수요지 위도': demand_lat,
                '수요지 경도': demand_lon,
                '후보지 번호': j,
                '후보지 위도': candidate_lat,
                '후보지 경도': candidate_lon
            })

# 전체 출력 설정
pd.set_option('display.max_rows', None)  # 모든 행 출력
pd.set_option('display.max_columns', None)  # 모든 열 출력
pd.set_option('display.expand_frame_repr', False)  # 한 줄로 출력되도록 설정

# 결과를 데이터프레임으로 변환
result_df = pd.DataFrame(results)

# 데이터프레임 출력
print(result_df)


In [ ]:
pd.DataFrame(result_df).to_excel('선정된_수요지_후보지_좌표.xlsx', index=False)

In [ ]:
# Tmap API Key
tmap_api_key = '' # 여기에 발급받은 TMAP API 키를 입력하세요.

# 데이터프레임에서 수요지와 후보지 정보를 가져오기
# 예를 들어 result_df라는 데이터프레임을 사용

# Folium 지도 생성
m = folium.Map(location=[result_df['수요지 위도'].mean(), result_df['수요지 경도'].mean()], zoom_start=14)

# 후보지 색상 리스트
colors = ['blue', 'green', 'purple', 'yellow', 'red', 'orange']

# 후보지 색상을 매핑할 딕셔너리
color_map = {}

# 각 수요지와 할당된 후보지에 대해 경로 요청 및 시각화
for idx, row in result_df.iterrows():
    start_lat = row['수요지 위도']
    start_lon = row['수요지 경도']
    end_lat = row['후보지 위도']
    end_lon = row['후보지 경도']
    후보지_번호 = row['후보지 번호']

    # 색상 맵에 후보지 번호가 없으면 색상 추가
    if 후보지_번호 not in color_map:
        color_map[후보지_번호] = colors[len(color_map) % len(colors)]

    # Tmap 경로 요청 URL
    url = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1&format=json"

    headers = {
        "Accept": "application/json",
        "appKey": tmap_api_key
    }

    # 경로 요청 파라미터
    params = {
        "startX": start_lon,
        "startY": start_lat,
        "endX": end_lon,
        "endY": end_lat,
        "reqCoordType": "WGS84GEO",
        "resCoordType": "EPSG3857",
        "startName": "출발지",
        "endName": "목적지"
    }

    # Tmap API 호출
    response = requests.post(url, headers=headers, json=params)  # data 대신 json 사용
    print(f"후보지 번호: {후보지_번호}, 상태 코드: {response.status_code}")  # 상태 코드 출력
    if response.status_code == 200:
        data = response.json()
        print(data)  # 반환된 데이터 출력
    else:
        print("API 요청 실패:", response.text)  # 에러 메시지 출력



    # EPSG3857 좌표계를 WGS84 좌표계로 변환
    transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326")

    wgs84_coordinates = []

    for feature in data['features']:
        coords = feature['geometry']['coordinates']
        if feature['geometry']['type'] == "LineString":
            for coord in coords:
                lat, lon = transformer.transform(coord[0], coord[1])
                wgs84_coordinates.append([lat, lon])

    # 출발지와 목적지 마크 추가
    folium.Marker(
        location=[end_lat, end_lon],
        popup=f"후보지 {후보지_번호}",
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(m)

    # 경로 추가 (색상 지정)
    if wgs84_coordinates:  # Ensure coordinates list is not empty
        color = color_map[후보지_번호]  # 후보지 번호에 따른 색상 선택
        folium.PolyLine(locations=wgs84_coordinates, color=color).add_to(m)

# 지도 저장
m.save("tmap_P-median_route.html")


# 8. 기존권역과 최적권역의 폐의약품 수거함까지 거리

In [ ]:
# 엑셀 파일에서 데이터 읽기
# 최적
result_df = pd.read_excel(r'선정된_수요지_후보지_좌표.xlsx')  # 수요지 데이터

distance_matrix = pd.read_excel('distance_matrix_TMAP(1-196).xlsx') # 후보지와 수요지 거리행렬 데이터

# 기존
result2_df = pd.read_excel(r'기존_수요지_후보지_좌표.xlsx')  # 수요지 데이터

distance2_matrix = pd.read_excel('distance_matrix_TMAP_기존.xlsx') # 기존 수거함과 수요지 거리행렬 데이터

In [ ]:
# 최적
# 수요지 번호와 후보지 번호 추출
demand_locations = result_df['수요지 번호'].values - 1  # 0부터 시작하도록 조정
selected_locations_pm = result_df['후보지 번호'].values  # 0부터 시작하도록 조정


# 각 수요지와 할당된 후보지 간의 거리 합 초기화
total_distance = 0


# 각 수요지에 대해 할당된 후보지와의 거리를 계산하고 합산
for demand_idx, facility_idx in zip(demand_locations, selected_locations_pm):
    total_distance += distance_matrix.iloc[demand_idx][facility_idx]


# 수요지의 수로 나누어 평균 거리 계산(수요지당 폐의약품 수거함까지의 평균 거리)
average_distance_selected_demand = total_distance / len(demand_locations)


# 후보지의 수로 나누어 평균 거리 계산(폐의약품 수거함당 수요지까지의 평균 거리)
average_distance_selected_pm = total_distance / len(list(set(selected_locations_pm)))


# 결과 출력
print("최적_수요지당 폐의약품 수거함까지의 평균 거리:", average_distance_selected_demand)
print("최적_폐의약품 수거함당 수요지까지의 평균 거리:", average_distance_selected_pm)

In [ ]:
# 수요지 번호와 후보지 번호 추출
demand_locations = result2_df['수요지 번호'].values - 1  # 0부터 시작하도록 조정
selected_locations_pm = result2_df['기존 후보지 번호'].values  # 0부터 시작하도록 조정


# 각 수요지와 할당된 후보지 간의 거리 합 초기화
total_distance = 0


# 각 수요지에 대해 할당된 후보지와의 거리를 계산하고 합산
for demand_idx, facility_idx in zip(demand_locations, selected_locations_pm):
    total_distance += distance2_matrix.iloc[demand_idx][facility_idx]


# 수요지의 수로 나누어 평균 거리 계산(수요지당 폐의약품 수거함까지의 평균 거리)
average_distance_selected_demand = total_distance / len(demand_locations)


# 후보지의 수로 나누어 평균 거리 계산(폐의약품 수거함당 수요지까지의 평균 거리)
average_distance_selected_pm = total_distance / len(list(set(selected_locations_pm)))


# 결과 출력
print("기존_수요지당 폐의약품 수거함까지의 평균 거리:", average_distance_selected_demand* 1000)
print("기존_폐의약품 수거함당 수요지까지의 평균 거리:", average_distance_selected_pm * 1000)

# 9. 선정된 폐의약품 수거함이 커버하는 세대 수

In [ ]:
# 수요지-선정된 후보지 df에 세대수 데이터 추가
result_df['세대 수'] = demand_weights
result_df

In [ ]:
# '후보지 번호'를 기준으로 '세대 수' 열의 합계를 구하기
sum_df = result_df.groupby('후보지 번호')['세대 수'].sum().reset_index()
sum_df

# 10. 기존의 폐의약품 수거함이 커버하는 세대 수

In [ ]:
# 엑셀 파일에서 데이터 읽기
distance2_df = pd.read_excel(r'distance_matrix_TMAP_기존.xlsx')  # 기존 거리 데이터
candidates2_df = pd.read_excel(r'기존 폐의약품 수거함 위치.xlsx')  # 기존 수거함 데이터

# 기존 후보지의 위도 경도 정보 추출
candidate2_locations = candidates2_df[['위도', '경도']].values

**기존 폐의약품 수거함의 P-median**

In [ ]:
# 후보지와 수요지의 좌표를 튜플 리스트로 변환
candidate2_coords = list(zip(candidate2_locations[:, 0], candidate2_locations[:, 1]))
demand_coords = list(zip(demand_locations[:, 0], demand_locations[:, 1]))

# p-median 문제 정의
p = 3  # 선택할 후보지의 수
prob_pm = LpProblem("P-Median_Problem", LpMinimize)

# 결정 변수 생성
use_location = LpVariable.dicts("Use_Location", range(len(candidate2_coords)), cat='Binary')
assign = LpVariable.dicts("Assign", [(i, j) for i in range(len(demand_coords)) for j in range(len(candidate2_coords))], cat='Binary')


# 목적 함수: 수요지에서 선택된 후보지까지의 거리 최소화
prob_pm += lpSum(demand_weights[i] * distance_df[j][i] * assign[(i, j)] for i in range(len(demand_coords)) for j in range(len(candidate2_coords)))

# 제약 조건: 정확히 p개의 후보지만 사용
prob_pm += lpSum(use_location[j] for j in range(len(candidate2_coords))) == p

# 제약 조건: 각 수요지는 하나의 후보지에만 할당
for i in range(len(demand_coords)):
    prob_pm += lpSum(assign[(i, j)] for j in range(len(candidate2_coords))) == 1

# 제약 조건: 수요지는 선택된 후보지에만 할당 가능
for i in range(len(demand_coords)):
    for j in range(len(candidate2_coords)):
        prob_pm += assign[(i, j)] <= use_location[j]

# 문제를 풀고 결과 출력
prob_pm.solve()

# 선택된 후보지 출력
selected_locations_pm = [j for j in range(len(candidate2_coords)) if use_location[j].value() == 1]
print("Selected candidate locations (P-Median):", selected_locations_pm)

# 각 수요지에 할당된 후보지 출력
for i in range(len(demand_coords)):
    for j in selected_locations_pm:
        if assign[(i, j)].value() == 1:
            print(f"Demand point {i+1} is assigned to candidate location {j}.")

In [ ]:
# 수요지와 후보지의 위도 경도 정보를 포함한 리스트를 생성
results = []

# 각 수요지에 할당된 후보지 정보를 추가
for i in range(len(demand_coords)):
    for j in selected_locations_pm:
        if assign[(i, j)].value() == 1:
            # 수요지와 후보지의 위도 경도 정보를 가져오기
            demand_lat, demand_lon = demand_coords[i]
            candidate_lat, candidate_lon = candidate2_coords[j]
            results.append({
                '수요지 번호': i + 1,
                '수요지 위도': demand_lat,
                '수요지 경도': demand_lon,
                '기존 후보지 번호': j,
                '기존 후보지 위도': candidate_lat,
                '기존 후보지 경도': candidate_lon
            })

# 전체 출력 설정
pd.set_option('display.max_rows', None)  # 모든 행 출력
pd.set_option('display.max_columns', None)  # 모든 열 출력
pd.set_option('display.expand_frame_repr', False)  # 한 줄로 출력되도록 설정

# 결과를 데이터프레임으로 변환
result2_df = pd.DataFrame(results)

# 데이터프레임 출력
print(result2_df)

**기존 폐의약품 수거함이 커버하는 세대 수의 합 구하기**

In [ ]:
# 수요지-선정된 후보지 df에 세대수 데이터 추가
result2_df['세대 수'] = demand_weights
result2_df

In [ ]:
# '후보지 번호'를 기준으로 '세대 수' 열의 합계를 구하기
sum2_df = result2_df.groupby('기존 후보지 번호')['세대 수'].sum().reset_index()
sum2_df